# Merge Raw Outputs into Per-Step Tables — GPT-5

Flattens the raw per-iteration JSON written by the pipeline runner
(`results/gpt5/raw/<conversation>/`, git-ignored) into per-cell CSV tables — one row per
(conversation × technique × iteration × participant [× restaurant]) — with the gold label joined
onto every row (`Answer` / `*_answer` columns).

These CSVs are shipped with the repository under `results/gpt5/`, so the model's raw behaviour can be
inspected without re-running the pipeline. They are also the data behind the Step 1.2 confusion
matrices (paper Figure 7).

| Output file | Content |
|---|---|
| `Step1_1_merged.csv` | participant / restaurant lists and chosen restaurant per iteration, with match rates |
| `Step1_2_Suggestion.csv`, `Step1_2_Response.csv` | per-participant egocentrism labels + the model's reasoning text |
| `Step2_merged.csv` | Mentioned-Table cells |
| `Step3_merged.csv` | Perception-Table cells |
| `Step4_merged.csv` | preference / constraint factor sets + their integrated union |

## Step 1 — entity lists and egocentrism

Loads every `Step1_<technique>_normalized_<iteration>.json`, joins the gold lists
(`P_answer`, `R_answer`, `F_answer`) and computes simple overlap rates (restaurant names are
compared after stripping the parenthetical readings, e.g. `（ジンディンロウ）`). The
suggestion / response tables keep the model's per-participant `reasoning` so individual label
decisions can be audited.

In [1]:
import os
import json
import pandas as pd
import re

# Path settings: change to your actual paths.
results_dir = 'results/gpt5/raw'
gold_dir    = 'data/gold'  # e.g.: 'data/gold'

# Initialize a list for storing results
records_step1_1 = []
records_sugg     = []
records_resp     = []

# --- Load STEP1.1 gold data --- #
gold1_map = {}
for folder in os.listdir(gold_dir):
    folder_path = os.path.join(gold_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id    = folder.replace('_log', '')
        gold1_path = os.path.join(folder_path, 'step1_1_gold.json')
        if os.path.isfile(gold1_path):
            with open(gold1_path, 'r', encoding='utf-8') as f:
                gd = json.load(f)
            gold1_map[conv_id] = {
                'P_answer': set(gd.get('participants', [])),
                'R_answer': set(gd.get('restaurant_brands', [])),
                'F_answer': gd.get('final_restaurant', '')
            }

# --- Load STEP1.2 gold data --- #
gold2_map_sugg = {}
gold2_map_resp = {}
for folder in os.listdir(gold_dir):
    folder_path = os.path.join(gold_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id    = folder.replace('_log', '')
        gold2_path = os.path.join(folder_path, 'step1_2_gold.json')
        if os.path.isfile(gold2_path):
            with open(gold2_path, 'r', encoding='utf-8') as f:
                gd2 = json.load(f)
            gold2_map_sugg[conv_id] = {
                e['participant']: e['suggestion_type']
                for e in gd2.get('suggestion_table', [])
            }
            gold2_map_resp[conv_id] = {
                e['participant']: e['response_type']
                for e in gd2.get('response_table', [])
            }

# --- Load and match prediction data --- #
normalize = lambda s: re.sub(r'（.*?）', '', s)

for folder in os.listdir(results_dir):
    if not folder.endswith('_log'):
        continue
    conv_id     = folder.replace('_log', '')
    folder_path = os.path.join(results_dir, folder)
    
    # Gold value
    gold1 = gold1_map.get(conv_id, {})
    P_ans = gold1.get('P_answer', set())
    R_ans = gold1.get('R_answer', set())
    F_ans = gold1.get('F_answer', '')
    
    # Process STEP1.1
    for technique in ['ND', 'CoT', 'ZS']:
        for iteration in range(1, 6):
            pred_path = os.path.join(
                folder_path, f'Step1_{technique}_normalized_{iteration}.json'
            )
            if not os.path.isfile(pred_path):
                continue
            with open(pred_path, 'r', encoding='utf-8') as f:
                pd1 = json.load(f)
            
            P_pred = set(pd1.get('participants', []))
            R_pred = set(pd1.get('restaurant_brands', []))
            F_pred = pd1.get('final_restaurant', '')
            
            # Compute the match rate
            p_match_rate = (
                len(P_pred & P_ans) / len(P_ans)
                if P_ans else None
            )
            R_pred_norm = set(normalize(r) for r in R_pred)
            R_ans_norm  = set(normalize(r) for r in R_ans)
            r_match_rate = (
                len(R_pred_norm & R_ans_norm) / len(R_ans_norm)
                if R_ans_norm else None
            )
            # Final Match Rate
            match_rate = (
                p_match_rate * r_match_rate
                if p_match_rate is not None and r_match_rate is not None
                else None
            )
            
            records_step1_1.append({
                'conversation_id':   conv_id,
                'technique':        technique,
                'iteration':        iteration,
                'participants':     '|'.join(sorted(P_pred)),
                'P_answer':         '|'.join(sorted(P_ans)),
                'p_match_rate':     p_match_rate,
                'restaurant_brands':'|'.join(sorted(R_pred)),
                'R_answer':         '|'.join(sorted(R_ans)),
                'r_match_rate':     r_match_rate,
                'final_restaurant': F_pred,
                'F_answer':         F_ans,
                'match_rate':       match_rate
            })
            
            # Process STEP1.2 Suggestion & Response
            for s in pd1.get('suggestion_table', []):
                participant = s.get('participant', '')
                records_sugg.append({
                    'conversation_id': conv_id,
                    'technique':       technique,
                    'iteration':       iteration,
                    'participant':     participant,

                    'suggestion':      s.get('suggestion_type', ''),
                    'Answer':          gold2_map_sugg.get(conv_id, {}).get(participant, ''),
                    'reasoning':       s.get('reasoning', '')
                })
            for r in pd1.get('response_table', []):
                participant = r.get('participant', '')
                records_resp.append({
                    'conversation_id': conv_id,
                    'technique':       technique,
                    'iteration':       iteration,
                    'participant':     participant,

                    'response':        r.get('response_type', ''),
                    'Answer':          gold2_map_resp.get(conv_id, {}).get(participant, ''),
                    'reasoning':       r.get('reasoning', '')
                })

# --- Build DataFrame and save CSV --- #
# Step1.1 results
df1 = pd.DataFrame(records_step1_1)
df1.to_csv('results/gpt5/Step1_1_merged.csv', index=False, encoding='utf-8-sig')

# Step1.2 Suggestion
df_sugg = pd.DataFrame(records_sugg)
df_sugg.to_csv('results/gpt5/Step1_2_Suggestion.csv', index=False, encoding='utf-8-sig')

# Step1.2 Response
df_resp = pd.DataFrame(records_resp)
df_resp.to_csv('results/gpt5/Step1_2_Response.csv', index=False, encoding='utf-8-sig')

# Preview the results
print(df1.head())
print(df_sugg.head())
print(df_resp.head())


  conversation_id technique  iteration participants P_answer  p_match_rate  \
0               9        ND          1      A|B|C|D  A|B|C|D           1.0   
1               9        ND          2      A|B|C|D  A|B|C|D           1.0   
2               9        ND          3      A|B|C|D  A|B|C|D           1.0   
3               9        ND          4      A|B|C|D  A|B|C|D           1.0   
4               9        ND          5      A|B|C|D  A|B|C|D           1.0   

                                   restaurant_brands  \
0  Nihonbashi 1-1-1|おでんやden（おでんやでん）|北丸 新宿南口店 （きたま...   
1  おでんやden（おでんやでん）|北丸 新宿南口店 （きたまる）|博多串焼き・野菜巻き工房 渋...   
2  おでんやden（おでんやでん）|北丸 新宿南口店 （きたまる）|博多串焼き・野菜巻き工房 渋...   
3  Nihonbashi 1-1-1|おでんやden（おでんやでん）|北丸 新宿南口店 （きたま...   
4  おでんやden（おでんやでん）|北丸 新宿南口店 （きたまる）|博多串焼き・野菜巻き工房 渋...   

                                            R_answer  r_match_rate  \
0  Nihonbashi 1-1-1|おでんやden（おでんやでん）|北丸 新宿南口店 （きたま...      0.909091   
1  Nihonbashi 1-1-1|おでんやden（おでんやでん）|北丸 新宿南口店 （きたま...  

## Step 2 — Mentioned-Table cells

One row per (participant × restaurant) cell per technique × iteration; `Answer` is the gold
mention label joined by exact (participant, restaurant) match.

In [2]:
import os
import json
import pandas as pd

# Path settings: change to your actual paths.
results_dir = 'results/gpt5/raw' 
gold_dir = 'data/gold'     # e.g.: 'data/gold'

records = []

# Build Gold mapping: (conv_id, participant, restaurant) -> mention gold
gold_map = {}
for folder in os.listdir(gold_dir):
    folder_path = os.path.join(gold_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id = folder.replace('_log', '')
        gold_path = os.path.join(folder_path, 'step2_gold.json')
        if os.path.isfile(gold_path):
            with open(gold_path, 'r', encoding='utf-8') as f:
                gold_data = json.load(f)
            for entry in gold_data.get('mentioned_table', []):
                key = (conv_id, entry.get('participant', ''), entry.get('restaurant', ''))
                gold_map[key] = entry.get('mention', '')

# Load and merge Predicted Step2 JSON files
for folder in os.listdir(results_dir):
    folder_path = os.path.join(results_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id = folder.replace('_log', '')
        
        for technique in ['CoT', 'SR', 'PD', 'MoRE']:
            for iteration in range(1, 6):
                fname = f'{conv_id}_log_Step2_{technique}_{iteration}.json'
                fpath = os.path.join(folder_path, fname)
                if os.path.isfile(fpath):
                    with open(fpath, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    mention_list = data.get('Result', {}).get('mentioned_table', [])
                    f1_mentioned = data.get('Detailed_Performance', {}).get('F1_Mentioned', None)
                    
                    for entry in mention_list:
                        participant = entry.get('participant', '')
                        restaurant = entry.get('restaurant', '')
                        pred_mention = entry.get('mention', '')
                        answer = gold_map.get((conv_id, participant, restaurant), '')
                        
                        records.append({
                            'conversation_id': conv_id,
                            'technique': technique,
                            'iteration': iteration,
                            'participant': participant,
                            'restaurant': restaurant,
                            'mention': pred_mention,
                            'Answer': answer,
                            # 'F1_Mentioned': f1_mentioned
                        })

# Build DataFrame and save CSV
df = pd.DataFrame(records)

# Specify column order
cols = [
    'conversation_id', 'technique', 'iteration',
    'participant', 'restaurant', 'mention', 'Answer'#, 'F1_Mentioned'
]
df = df[cols]

output_file = 'results/gpt5/Step2_merged.csv'


df.to_csv(output_file,
          index=False,
          encoding='utf-8-sig',
          escapechar='\\')   # specify backslash as the escape char

# Check results
df.head()


,conversation_id,technique,iteration,participant,restaurant,mention,Answer
0,9,CoT,1,A,おでんやden（おでんやでん）,Mentioned,Mentioned
1,9,CoT,1,A,純米酒専門YATA 新宿三丁目店（ヤタ）,None,None
2,9,CoT,1,A,博多串焼き・野菜巻き工房 渋谷宮益坂のごりょんさん,None,None
3,9,CoT,1,A,赤から 新宿東口店,None,None
4,9,CoT,1,A,居酒屋 四谷 なごみ,Mentioned,Mentioned


## Step 3 — Perception-Table cells

Same layout as Step 2, with the model's `sentiment` label against the gold `perception` label.

In [3]:
import os
import json
import pandas as pd

# Path settings: change to your actual paths.
results_dir = 'results/gpt5/raw'
gold_dir = 'data/gold'  # e.g.: 'data/gold'

records = []

# Build Gold mapping: (conv_id, participant, restaurant) -> perception
gold_map = {}
for folder in os.listdir(gold_dir):
    folder_path = os.path.join(gold_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id = folder.replace('_log', '')
        gold_path = os.path.join(folder_path, 'step3_gold.json')
        if os.path.isfile(gold_path):
            with open(gold_path, 'r', encoding='utf-8') as f:
                gold_data = json.load(f)
            for entry in gold_data.get('perception_table', []):
                key = (conv_id, entry.get('participant', ''), entry.get('restaurant', ''))
                gold_map[key] = entry.get('perception', '')

# Load and merge Predicted JSON files
for folder in os.listdir(results_dir):
    folder_path = os.path.join(results_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id = folder.replace('_log', '')
        
        for technique in ['CoT', 'SR', 'PD', 'MoRE']:
            for iteration in range(1, 6):
                fname = f'{conv_id}_log_Step3_{technique}_{iteration}.json'
                fpath = os.path.join(folder_path, fname)
                if os.path.isfile(fpath):
                    with open(fpath, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    sentiment_list = data.get('Result', {}).get('sentiment_table', [])
                    f1_total = data.get('Detailed_Performance', {}).get('F1_total', None)
                    
                    for entry in sentiment_list:
                        participant = entry.get('participant', '')
                        restaurant = entry.get('restaurant', '')
                        sentiment = entry.get('sentiment', '')
                        answer = gold_map.get((conv_id, participant, restaurant), '')
                        
                        records.append({
                            'conversation_id': conv_id,
                            'technique': technique,
                            'iteration': iteration,
                            'participant': participant,
                            'restaurant': restaurant,
                            'sentiment': sentiment,
                            'Answer': answer,
                            # 'F1_total': f1_total
                        })

# Build DataFrame and save CSV
df = pd.DataFrame(records)

# Reorder columns
cols = [
    'conversation_id', 'technique', 'iteration',
    'participant', 'restaurant', 'sentiment', 'Answer'#, 'F1_total'
]
df = df[cols]

output_file = 'results/gpt5/Step3_merged.csv'
df.to_csv(output_file,
          index=False,
          encoding='utf-8-sig',
          escapechar='\\')   # specify backslash as the escape char

# Check results
df.head()


,conversation_id,technique,iteration,participant,restaurant,sentiment,Answer
0,9,CoT,1,A,おでんやden（おでんやでん）,Positive,Positive
1,9,CoT,1,A,純米酒専門YATA 新宿三丁目店（ヤタ）,Positive,Mix
2,9,CoT,1,A,博多串焼き・野菜巻き工房 渋谷宮益坂のごりょんさん,Neutral,Positive
3,9,CoT,1,A,赤から 新宿東口店,Positive,Positive
4,9,CoT,1,A,居酒屋 四谷 なごみ,Neutral,Neutral


## Step 4 — preference / constraint factors

Besides the raw `preferences` / `constraints` strings, this builds the `integrated` column —
the union of both factor sets per cell (and `integrated_answer` for gold). The integrated table
is what Positive-F1 evaluates (paper Eq. 5–6, Table 7).

In [4]:
import os
import json
import pandas as pd

# Path settings: change to your actual paths.
results_dir = 'results/gpt5/raw'
gold_dir    = 'data/gold'         # e.g.: 'data/gold'

records = []

# --- Load STEP4 gold data --- #
# (conv_id, participant, restaurant) -> (pre_answer, cons_answer)
gold_pre = {}
gold_cons = {}

for folder in os.listdir(gold_dir):
    folder_path = os.path.join(gold_dir, folder)
    if folder.endswith('_log') and os.path.isdir(folder_path):
        conv_id    = folder.replace('_log', '')
        gold4_path = os.path.join(folder_path, 'step4_gold.json')
        if os.path.isfile(gold4_path):
            with open(gold4_path, 'r', encoding='utf-8') as f:
                gd4 = json.load(f)
            # preference answers
            for e in gd4.get('preference_table', []):
                key = (conv_id, e['participant'], e['restaurant'])
                gold_pre[key] = e.get('preferences', '')
            # constraint answers
            for e in gd4.get('constraint_table', []):
                key = (conv_id, e['participant'], e['restaurant'])
                gold_cons[key] = e.get('constraints', '')

# --- Load and merge STEP4 prediction data --- #
for folder in os.listdir(results_dir):
    if not folder.endswith('_log'):
        continue
    conv_id     = folder.replace('_log', '')
    folder_path = os.path.join(results_dir, folder)
    
    for technique in ['CoT', 'SR', 'PD', 'MoRE']:
        for iteration in range(1, 6):
            fname = f'{conv_id}_log_Step4_{technique}_{iteration}.json'
            fpath = os.path.join(folder_path, fname)
            if not os.path.isfile(fpath):
                continue
            with open(fpath, 'r', encoding='utf-8') as f:
                pd4 = json.load(f)
            
            pref_list = pd4.get('Result', {}).get('preference_table', [])
            cons_list = pd4.get('Result', {}).get('constraint_table', [])
            cons_map = {
                (c['participant'], c['restaurant']): c.get('constraints','')
                for c in cons_list
            }
            
            for p in pref_list:
                participant = p.get('participant','')
                restaurant  = p.get('restaurant','')
                preferences = p.get('preferences','').strip()
                pre_ans     = gold_pre.get((conv_id, participant, restaurant), '').strip()
                cons_pred   = cons_map.get((participant, restaurant), '').strip()
                cons_ans    = gold_cons.get((conv_id, participant, restaurant), '').strip()
                
                # ---------- Compute integrated column ---------- #
                # First check whether the original is exactly "None"
                pref_is_none = (preferences.lower() == 'none')
                cons_is_none = (cons_pred.lower() == 'none')
                
                if pref_is_none and cons_is_none:
                    # If both are "None", keep "None" as is
                    integrated = 'None'
                else:
                    # Otherwise, build the union excluding "None" entries
                    pref_items = [
                        x.strip() for x in preferences.split(',')
                        if x.strip() and x.strip().lower() != 'none'
                    ] if preferences else []
                    
                    cons_items = [
                        x.strip() for x in cons_pred.split(',')
                        if x.strip() and x.strip().lower() != 'none'
                    ] if cons_pred else []
                    
                    integrated_list = []
                    for item in pref_items:
                        if item not in integrated_list:
                            integrated_list.append(item)
                    for item in cons_items:
                        if item not in integrated_list:
                            integrated_list.append(item)
                    
                    integrated = ','.join(integrated_list) if integrated_list else ''
                
                # ---------- Compute gold integrated column ---------- #
                ans_pref_is_none = (pre_ans.lower() == 'none')
                ans_cons_is_none = (cons_ans.lower() == 'none')
                
                if ans_pref_is_none and ans_cons_is_none:
                    integrated_answer = 'None'
                else:
                    ans_pref_items = [
                        x.strip() for x in pre_ans.split(',')
                        if x.strip() and x.strip().lower() != 'none'
                    ] if pre_ans else []
                    
                    ans_cons_items = [
                        x.strip() for x in cons_ans.split(',')
                        if x.strip() and x.strip().lower() != 'none'
                    ] if cons_ans else []
                    
                    integrated_ans_list = []
                    for item in ans_pref_items:
                        if item not in integrated_ans_list:
                            integrated_ans_list.append(item)
                    for item in ans_cons_items:
                        if item not in integrated_ans_list:
                            integrated_ans_list.append(item)
                    
                    integrated_answer = (
                        ','.join(integrated_ans_list) 
                        if integrated_ans_list 
                        else ''
                    )
                
                records.append({
                    'conversation_id':    conv_id,
                    'technique':          technique,
                    'iteration':          iteration,
                    'participant':        participant,
                    'restaurant':         restaurant,
                    'preferences':        preferences,
                    'pre_answer':         pre_ans,
                    'constraints':        cons_pred,
                    'cons_answer':        cons_ans,
                    'integrated':         integrated,
                    'integrated_answer':  integrated_answer
                })

# Build and save DataFrame
df4 = pd.DataFrame(records)
output_file = 'results/gpt5/Step4_merged.csv'
df4.to_csv(output_file,
           index=False,
           encoding='utf-8-sig',
           escapechar='\\')

# Check results (example)
print(df4.head())


  conversation_id technique  iteration participant  \
0               9       CoT          1           A   
1               9       CoT          1           A   
2               9       CoT          1           A   
3               9       CoT          1           A   
4               9       CoT          1           A   

                   restaurant  preferences pre_answer constraints cons_answer  \
0             おでんやden（おでんやでん）  A1,A2,A4,A7         A7        None        None   
1        純米酒専門YATA 新宿三丁目店（ヤタ）        A1,A7      A7,A1          A2          A2   
2  博多串焼き・野菜巻き工房  渋谷宮益坂のごりょんさん         None         A7        None        None   
3                   赤から 新宿東口店           A1         A7        None        None   
4                  居酒屋 四谷 なごみ        A2,A7       None        None        None   

    integrated integrated_answer  
0  A1,A2,A4,A7                A7  
1     A1,A7,A2          A7,A1,A2  
2         None                A7  
3           A1                A7  
4        A2,A